# Top Rated Coffee Dataset Wrangled

# Table of Contents — Top Rated Coffee Dataset Wrangled

1. Imports & Pathways    
2. Data Inspection  
3. Data Wrangle – Roaster Location  
4. Data Wrangle – Coffee Origin  
5. Wrangle Region
6. Wrangle Coffee Region
7. Wrangle Roaster Region
8. Export

## 1. Imports/Pathways

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import os
import re

In [2]:
# File Pathway
path = r"C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee"

In [3]:
# Import dataset
df_coffee_wrangle = pd.read_csv(
    r'C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee\02_Data\Prepared_Data\Top Rated Coffee\top_rated_coffee_clean.csv')

## 2. Data Inspection

In [4]:
df_coffee_wrangle.head()

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,agtron_roast,price_usd,quantity_g,usd_per_gram
0,Colombia Finca Campo Hermosa,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,82.0,29.99,226.8,$29.99 / 226.8g
1,Colombia Finca La Sirena Mango Co-Ferment,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,87.0,22.99,226.8,$22.99 / 226.8g
2,In Bloom,94.0,"Jersey City, New Jersey",Colombia; Ethiopia,Light,88.0,25.00,250.0,$25.0 / 250.0g
3,Ethiopia Washed Kaffa Gimbo Lot Rich Espresso,96.0,"Chia-Yi, Taiwan","Gimbo, Kaffa Province, Ethiopia",Medium Light,77.0,8.17,226.8,$8.17 / 226.8g
4,Ethiopia Natural Gute Bona,95.0,"Chia-Yi, Taiwan","Sidamo growing region, southern Ethiopia",Medium Light,78.0,13.07,226.8,$13.07 / 226.8g


## 3. Data Wrangle - Roaster Location

In [5]:
# Temporarily expand row display limit
pd.set_option('display.max_rows', None)

# Show all unique locations
pd.DataFrame({'roaster_location': sorted(df_coffee_wrangle['roaster_location'].dropna().unique())})

,roaster_location
0,"Acton, Massachusetts"
1,"Albuquerque, New Mexico"
2,"Anchorage, Alaska"
3,"Annapolis, Maryland"
4,"Anoka, Minnesota"
5,"Antigua Guatemala, Guatemala"
6,"Antigua, Guatemala"
7,"Arcata, California"
8,"Arlington, Massachusetts"
9,"Ashland, Oregon"


In [6]:
# Create the inital splits
split_cols = df_coffee_wrangle['roaster_location'].str.split(',', expand=True)

# Assign based on how many parts exist
df_coffee_wrangle['roaster_city'] = split_cols[0].str.strip()
df_coffee_wrangle['roaster_region'] = split_cols[1].str.strip() if split_cols.shape[1] > 1 else None
df_coffee_wrangle['roaster_country'] = split_cols[2].str.strip() if split_cols.shape[1] > 2 else None

In [7]:
df_coffee_wrangle[['roaster_location', 'roaster_city', 'roaster_region', 'roaster_country']].head(20)

,roaster_location,roaster_city,roaster_region,roaster_country
0,"Cleveland, Tennessee",Cleveland,Tennessee,None
1,"Cleveland, Tennessee",Cleveland,Tennessee,None
2,"Jersey City, New Jersey",Jersey City,New Jersey,None
3,"Chia-Yi, Taiwan",Chia-Yi,Taiwan,None
4,"Chia-Yi, Taiwan",Chia-Yi,Taiwan,None
5,"Taichung, Taiwan",Taichung,Taiwan,None
6,"Yunlin, Taiwan",Yunlin,Taiwan,None
7,"Chia-Yi, Taiwan",Chia-Yi,Taiwan,None
8,"Chia-Yi, Taiwan",Chia-Yi,Taiwan,None
9,"Yunlin, Taiwan",Yunlin,Taiwan,None


In [8]:
# Countries list
known_countries = sorted([
    'Australia', 'Brazil', 'Burundi', 'Canada', 'China', 'Colombia', 'Costa Rica',
    'El Salvador', 'England', 'Ethiopia', 'Guatemala', 'Guinea', 'Honduras',
    'Hong Kong', 'India', 'Indonesia', 'Japan', 'Kenya', 'Macao', 'Mexico',
    'Nicaragua', 'Panama', 'Peru', 'Puerto Rico', 'South Korea', 'Taiwan',
    'Tanzania', 'Timor', 'United Kingdom', 'United States', 'Yemen'
])

In [9]:
# If a value in roaster_region is actually a country, move it to roaster_country
df_coffee_wrangle['roaster_country'] = df_coffee_wrangle.apply(
    lambda row: row['roaster_region'] if row['roaster_region'] in known_countries else row['roaster_country'],
    axis=1
)

In [10]:
# Work check
# Preview promoted country values without modifying the original column
df_coffee_wrangle['roaster_country_preview'] = df_coffee_wrangle.apply(
    lambda row: row['roaster_region'] if row['roaster_region'] in known_countries else row['roaster_country'],
    axis=1
)

# Display side-by-side comparison
df_coffee_wrangle[['roaster_location', 'roaster_region', 'roaster_country', 'roaster_country_preview']].head(20)

,roaster_location,roaster_region,roaster_country,roaster_country_preview
0,"Cleveland, Tennessee",Tennessee,None,None
1,"Cleveland, Tennessee",Tennessee,None,None
2,"Jersey City, New Jersey",New Jersey,None,None
3,"Chia-Yi, Taiwan",Taiwan,Taiwan,Taiwan
4,"Chia-Yi, Taiwan",Taiwan,Taiwan,Taiwan
5,"Taichung, Taiwan",Taiwan,Taiwan,Taiwan
6,"Yunlin, Taiwan",Taiwan,Taiwan,Taiwan
7,"Chia-Yi, Taiwan",Taiwan,Taiwan,Taiwan
8,"Chia-Yi, Taiwan",Taiwan,Taiwan,Taiwan
9,"Yunlin, Taiwan",Taiwan,Taiwan,Taiwan


In [11]:
# If country was moved over, remove duplicate from the roaster_region column
df_coffee_wrangle['roaster_region'] = df_coffee_wrangle.apply(
    lambda row: None if row['roaster_region'] in known_countries else row['roaster_region'],
    axis=1
)

In [12]:
# Work check
df_coffee_wrangle[['roaster_location', 'roaster_region', 'roaster_country']].head(20)

,roaster_location,roaster_region,roaster_country
0,"Cleveland, Tennessee",Tennessee,None
1,"Cleveland, Tennessee",Tennessee,None
2,"Jersey City, New Jersey",New Jersey,None
3,"Chia-Yi, Taiwan",None,Taiwan
4,"Chia-Yi, Taiwan",None,Taiwan
5,"Taichung, Taiwan",None,Taiwan
6,"Yunlin, Taiwan",None,Taiwan
7,"Chia-Yi, Taiwan",None,Taiwan
8,"Chia-Yi, Taiwan",None,Taiwan
9,"Yunlin, Taiwan",None,Taiwan


Looks like it worked moving over the countries to the roaster_country column and removing the duplicates in the roaster region. Later the None will become unknown.

In [13]:
# Address the us states 
us_states = [
    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut',
    'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa',
    'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan',
    'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
    'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio',
    'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota',
    'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia',
    'Wisconsin', 'Wyoming'
]

In [14]:
#  Fill in roaster_country Where Region Is a U.S. State
df_coffee_wrangle['roaster_country'] = df_coffee_wrangle.apply(
    lambda row: 'United States' if row['roaster_region'] in us_states and pd.isna(row['roaster_country']) else row['roaster_country'],
    axis=1
)

In [15]:
# Work check
df_coffee_wrangle[['roaster_location', 'roaster_region', 'roaster_country']].head(20)

,roaster_location,roaster_region,roaster_country
0,"Cleveland, Tennessee",Tennessee,United States
1,"Cleveland, Tennessee",Tennessee,United States
2,"Jersey City, New Jersey",New Jersey,United States
3,"Chia-Yi, Taiwan",None,Taiwan
4,"Chia-Yi, Taiwan",None,Taiwan
5,"Taichung, Taiwan",None,Taiwan
6,"Yunlin, Taiwan",None,Taiwan
7,"Chia-Yi, Taiwan",None,Taiwan
8,"Chia-Yi, Taiwan",None,Taiwan
9,"Yunlin, Taiwan",None,Taiwan


In [ ]:
# Drop these columns used for splitting and previewing
df_coffee_wrangle.drop(columns=[
    'origin_part_1',
    'origin_part_2',
    'origin_part_3',
    'roaster_country_preview'
], inplace=True)

In [16]:
df_coffee_wrangle.head()

,coffee_name,total_score,roaster_location,coffee_origin,roast_level,agtron_roast,price_usd,quantity_g,usd_per_gram,roaster_city,roaster_region,roaster_country,roaster_country_preview
0,Colombia Finca Campo Hermosa,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,82.0,29.99,226.8,$29.99 / 226.8g,Cleveland,Tennessee,United States,None
1,Colombia Finca La Sirena Mango Co-Ferment,94.0,"Cleveland, Tennessee","Quindio Department, Colombia",Light,87.0,22.99,226.8,$22.99 / 226.8g,Cleveland,Tennessee,United States,None
2,In Bloom,94.0,"Jersey City, New Jersey",Colombia; Ethiopia,Light,88.0,25.00,250.0,$25.0 / 250.0g,Jersey City,New Jersey,United States,None
3,Ethiopia Washed Kaffa Gimbo Lot Rich Espresso,96.0,"Chia-Yi, Taiwan","Gimbo, Kaffa Province, Ethiopia",Medium Light,77.0,8.17,226.8,$8.17 / 226.8g,Chia-Yi,None,Taiwan,Taiwan
4,Ethiopia Natural Gute Bona,95.0,"Chia-Yi, Taiwan","Sidamo growing region, southern Ethiopia",Medium Light,78.0,13.07,226.8,$13.07 / 226.8g,Chia-Yi,None,Taiwan,Taiwan


## 4. Data Wrangle - Coffee Origin 

In [17]:
# Get the list of places
# Expand display limit
pd.set_option('display.max_rows', None)

# Show all unique coffee_origin values
pd.DataFrame({'coffee_origin': sorted(df_coffee_wrangle['coffee_origin'].dropna().unique())})

,coffee_origin
0,"Acatenango growing region, Guatemala"
1,"Acatenango growing region, Guatemala."
2,"Acatenango growing region, central Guatemala"
3,"Acatenango growing region, central Guatemala."
4,"Acatenango, Chimaltenango Department, Guatemala"
5,"Acatenango, Chimaltenango, Guatemala"
6,"Acatenango, Guatemala"
7,"Aceh Province, Sumatra, Indonesia"
8,"Aceh Province, northern Sumatra Indonesia"
9,"Aceh Province, northern Sumatra, Indonesia."


In [18]:
# Define known countries
known_countries = sorted([
    'australia', 'brazil', 'burundi', 'canada', 'china', 'colombia', 'costa rica',
    'el salvador', 'england', 'ethiopia', 'guatemala', 'guinea', 'honduras',
    'hong kong', 'india', 'indonesia', 'japan', 'kenya', 'macao', 'mexico',
    'nicaragua', 'panama', 'peru', 'puerto rico', 'rwanda', 'south korea',
    'taiwan', 'tanzania', 'timor', 'united kingdom', 'united states', 'yemen'
])

In [19]:
# Split the coffee_origin string into parts using commas or semicolons
origin_split = df_coffee_wrangle['coffee_origin'].str.lower().str.strip().str.split(r'[;,]', expand=True)

In [20]:
# Assign each split part to a new column
df_coffee_wrangle.loc[:, 'origin_part_1'] = origin_split[0].str.strip()
df_coffee_wrangle.loc[:, 'origin_part_2'] = origin_split[1].str.strip() if origin_split.shape[1] > 1 else None
df_coffee_wrangle.loc[:, 'origin_part_3'] = origin_split[2].str.strip() if origin_split.shape[1] > 2 else None
df_coffee_wrangle.loc[:, 'origin_part_4'] = origin_split[3].str.strip() if origin_split.shape[1] > 3 else None

In [21]:
# Move over a known country from part_2 or part_3 if present
# Move over a known country from any origin part
# Initialize coffee_country before applying logic
df_coffee_wrangle['coffee_country'] = 'unknown'

# Infer country from origin parts
df_coffee_wrangle['coffee_country'] = df_coffee_wrangle.apply(
    lambda row: next((part for part in [
        row['origin_part_1'], row['origin_part_2'], row['origin_part_3'], row['origin_part_4']
    ] if part in known_countries), row['coffee_country']),
    axis=1
)

In [22]:
# If no country was moved over, scan all parts for known countries
df_coffee_wrangle.loc[:, 'coffee_country'] = df_coffee_wrangle.apply(
    lambda row: next((part for part in [
        row['origin_part_1'], row['origin_part_2'], row['origin_part_3'], row['origin_part_4']
    ] if part in known_countries), row['coffee_country']),
    axis=1
)

In [23]:
# Relabel headers
df_coffee_wrangle.loc[:, 'coffee_city'] = df_coffee_wrangle['origin_part_1']
df_coffee_wrangle.loc[:, 'coffee_region'] = df_coffee_wrangle['origin_part_2']

In [24]:
# Drop columns that aren't needed
columns_to_drop = ['origin_part_1', 'origin_part_2', 'origin_part_3', 'origin_part_4']
existing_cols = [col for col in columns_to_drop if col in df_coffee_wrangle.columns]
df_coffee_wrangle = df_coffee_wrangle.drop(columns=existing_cols)

In [25]:
# Add multi-orgin flag to prevent data removal
df_coffee_wrangle['multi_origin_flag'] = df_coffee_wrangle['coffee_origin'].str.contains(';', case=False, na=False)

In [26]:
# Clean coffee_city
df_coffee_wrangle.loc[
    (~df_coffee_wrangle['multi_origin_flag']) & df_coffee_wrangle['coffee_city'].isin(known_countries),
    'coffee_city'
] = None

In [27]:
# Clean coffee_region
df_coffee_wrangle.loc[
    (~df_coffee_wrangle['multi_origin_flag']) & df_coffee_wrangle['coffee_region'].isin(known_countries),
    'coffee_region'
] = None

In [28]:
# Missing values as unknown
location_cols = [
    'coffee_city', 'coffee_region', 'coffee_country',
    'roaster_city', 'roaster_region', 'roaster_country'
]

df_coffee_wrangle.loc[
    df_coffee_wrangle[location_cols].isna().all(axis=1),
    location_cols
] = 'unknown'

In [29]:
# Ensuring the new columns have a data type
df_coffee_wrangle = df_coffee_wrangle.astype({
    'coffee_city': 'string',
    'coffee_region': 'string',
    'coffee_country': 'string',
    'roaster_city': 'string',
    'roaster_region': 'string',
    'roaster_country': 'string'
})

In [30]:
#Cast location columns to string before filling
location_cols = [
    'coffee_city', 'coffee_region', 'coffee_country',
    'roaster_city', 'roaster_region', 'roaster_country'
]

# Cast to string to avoid dtype conflict
df_coffee_wrangle[location_cols] = df_coffee_wrangle[location_cols].astype('string')

# Now safely fill missing values
df_coffee_wrangle[location_cols] = df_coffee_wrangle[location_cols].fillna('unknown')

In [31]:
# Replace actual missing values (None/NaN) with 'unknown'
df_coffee_wrangle.fillna('unknown', inplace=True)

C:\Users\Chase\AppData\Local\Temp\ipykernel_16676\3533287444.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unknown' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_coffee_wrangle.fillna('unknown', inplace=True)


In [32]:
# Rearrange the columns for better flow
# Define final column order
final_columns = [
    'coffee_name',
    'total_score',
    'roast_level',
    'agtron_roast',
    'price_usd',
    'quantity_g',
    'usd_per_gram',
    'roaster_location',
    'roaster_city',
    'roaster_region',
    'roaster_country',
    'coffee_origin',
    'coffee_city',
    'coffee_region',
    'coffee_country'
]

# Reorder DataFrame
df_coffee_wrangle = df_coffee_wrangle[final_columns]

In [33]:
# Make text lowercase
# Apply lowercase to all object or string columns
for col in df_coffee_wrangle.columns:
    if df_coffee_wrangle[col].dtype == 'object' or df_coffee_wrangle[col].dtype.name == 'string':
        df_coffee_wrangle[col] = df_coffee_wrangle[col].astype(str).str.lower()

In [34]:
# Clean up data a bit
df_coffee_wrangle[string_cols] = df_coffee_wrangle[string_cols].apply(lambda col: col.str.strip())

NameError: name 'string_cols' is not defined

In [35]:
# Work check
df_coffee_wrangle.head(20)

,coffee_name,total_score,roast_level,agtron_roast,price_usd,quantity_g,usd_per_gram,roaster_location,roaster_city,roaster_region,roaster_country,coffee_origin,coffee_city,coffee_region,coffee_country
0,colombia finca campo hermosa,94.0,light,82.0,29.99,226.8,$29.99 / 226.8g,"cleveland, tennessee",cleveland,tennessee,united states,"quindio department, colombia",quindio department,unknown,colombia
1,colombia finca la sirena mango co-ferment,94.0,light,87.0,22.99,226.8,$22.99 / 226.8g,"cleveland, tennessee",cleveland,tennessee,united states,"quindio department, colombia",quindio department,unknown,colombia
2,in bloom,94.0,light,88.0,25.0,250.0,$25.0 / 250.0g,"jersey city, new jersey",jersey city,new jersey,united states,colombia; ethiopia,colombia,ethiopia,colombia
3,ethiopia washed kaffa gimbo lot rich espresso,96.0,medium light,77.0,8.17,226.8,$8.17 / 226.8g,"chia-yi, taiwan",chia-yi,unknown,taiwan,"gimbo, kaffa province, ethiopia",gimbo,kaffa province,ethiopia
4,ethiopia natural gute bona,95.0,medium light,78.0,13.07,226.8,$13.07 / 226.8g,"chia-yi, taiwan",chia-yi,unknown,taiwan,"sidamo growing region, southern ethiopia",sidamo growing region,southern ethiopia,unknown
5,white geisha blend espresso,95.0,medium light,78.0,39.0,226.8,$39.0 / 226.8g,"taichung, taiwan",taichung,unknown,taiwan,panama; ethiopia,panama,ethiopia,panama
6,yunlin natural geisha cpag-1,95.0,medium light,78.0,26.14,113.4,$26.14 / 113.4g,"yunlin, taiwan",yunlin,unknown,taiwan,"yunlin, taiwan",yunlin,unknown,taiwan
7,colombia natural huila eulises guzman,94.0,medium light,80.0,11.44,226.8,$11.44 / 226.8g,"chia-yi, taiwan",chia-yi,unknown,taiwan,"huila department, colombia",huila department,unknown,colombia
8,peru washed cusco buena vista soola sl09,94.0,medium light,77.0,9.8,113.4,$9.8 / 113.4g,"chia-yi, taiwan",chia-yi,unknown,taiwan,"cusco, peru",cusco,unknown,peru
9,yunlin natural geisha,94.0,light,84.0,19.61,113.4,$19.61 / 113.4g,"yunlin, taiwan",yunlin,unknown,taiwan,"yunlin, taiwan",yunlin,unknown,taiwan


## 5. Wrangle Region

In [36]:
# Strip all unicode, leave only letters
def strip_unicode(text):
    if pd.isna(text):
        return 'unknown'
    # Remove non-ASCII characters
    ascii_text = text.encode('ascii', 'ignore').decode('ascii')
    # Remove all punctuation, digits, and symbols — keep only letters and spaces
    clean_text = re.sub(r'[^a-zA-Z\s]', '', ascii_text)
    return clean_text.lower().strip()

## 6. Wrangle Coffee Region

##### Hawaii Normalization

In [37]:
# Strip unicode from coffee_region
df_coffee_wrangle['coffee_region'] = df_coffee_wrangle['coffee_region'].apply(strip_unicode)

In [38]:
# Preview work
df_coffee_wrangle['coffee_region'].value_counts(dropna=False).head(30)

coffee_region
unknown                        486
southern ethiopia              237
southcentral kenya             159
western panama                 123
oromia region                  113
southcentral ethiopia           72
big island of hawaii            51
north kona growing district     47
cauca department                42
yirgacheffe growing region      29
gedeo zone                      28
ethiopia                        27
central kenya                   26
guji zone                       26
huila department                24
guatemala                       22
north sumatra province          22
sumatra                         19
colombia                        18
volcan                          14
northern sumatra                14
jimma zone                      14
kenya                           13
boquete growing region          13
central highlands               12
southern colombia               12
chiriqui province               12
kona                            11
quindi

In [42]:
# Work check
df_coffee_wrangle[['coffee_origin', 'coffee_region']].head(50)

,coffee_origin,coffee_region
0,"quindio department, colombia",unknown
1,"quindio department, colombia",unknown
2,colombia; ethiopia,ethiopia
3,"gimbo, kaffa province, ethiopia",kaffa province
4,"sidamo growing region, southern ethiopia",southern ethiopia
5,panama; ethiopia,ethiopia
6,"yunlin, taiwan",unknown
7,"huila department, colombia",unknown
8,"cusco, peru",unknown
9,"yunlin, taiwan",unknown


In [44]:
# Pull Unique coffee_region Values
unique_regions = sorted(df_coffee_wrangle['coffee_region'].dropna().unique().tolist())
for region in unique_regions:
    print(region)

aberdare ridge
aceh province
acevedo
al hayma district
al kharijiyah district
alajuela
alajuela province
alta verapaz
alta verapaz department
amazonas region
ana sora district
antigua department
antioquia
antioquia department
apaneca growing region
arbegona
asia pacific
ataco
bench maji zone
bensa district
big island of hawaii
bolivia
boquete
boquete growing region
borena hagermariam district
brazil
brunca growing region
bule hora district
bure woreda
burundi
cajamarca
calda department
caldas
caldas department
cartago province
cauca
cauca department
central america
central colombia
central costa rica
central guatemala
central highlands
central kenya
central province
central valley
chaiyi
chiapas
chiapas state
chiayi
chiayi city
chiayi county
chimaltenango
chimaltenango department
chimborazo
chimborazo province
chiriqu
chiriqu province
chiriqui
chiriqui province
colombia
costa rica
cuilco
cundinamarca
cusco
democratic republic of the congo
dominican republic
east java
eastcentral colomb

In [45]:
# Filter hawaii related regions
hawaii_candidates = [
    region for region in unique_regions
    if any(keyword in region for keyword in ['hawaii', 'kona', 'kau', 'puna', 'lahaina'])
]

In [46]:
hawaii_map = {variant: 'hawaii' for variant in hawaii_candidates}

In [48]:
list(hawaii_map.keys())

['big island of hawaii',
 'hawaii',
 'hawaii island',
 'kau',
 'kona',
 'kona district',
 'kona growing region',
 'near lahaina',
 'north kona',
 'north kona district',
 'north kona growing district',
 'north kona growing region',
 'puna',
 'puna district',
 'southwestern corner of the big island of hawaii']

In [49]:
# ID and then apply hawaii normlization
def normalize_hawaii(region):
    return hawaii_map.get(region, region)

df_coffee_wrangle['coffee_region'] = df_coffee_wrangle['coffee_region'].apply(normalize_hawaii)

In [50]:
# Work check
df_coffee_wrangle['coffee_region'].value_counts(dropna=False).head(30)

coffee_region
unknown                       486
southern ethiopia             237
southcentral kenya            159
hawaii                        133
western panama                123
oromia region                 113
southcentral ethiopia          72
cauca department               42
yirgacheffe growing region     29
gedeo zone                     28
ethiopia                       27
guji zone                      26
central kenya                  26
huila department               24
north sumatra province         22
guatemala                      22
sumatra                        19
colombia                       18
volcan                         14
northern sumatra               14
jimma zone                     14
boquete growing region         13
kenya                          13
chiriqui province              12
southern colombia              12
central highlands              12
quindio department             10
valle del cauca                10
far western panama              9


In [51]:
# Spot Audit
df_coffee_wrangle[
    df_coffee_wrangle['coffee_region'].str.contains('hawaii|kona|kau|puna|lahaina', case=False, na=False)
    & (df_coffee_wrangle['coffee_region'] != 'hawaii')
][['coffee_origin', 'coffee_region']].head(30)

,coffee_origin,coffee_region


##### Strip Suffix

In [52]:
# Strip suffixes
suffixes_to_strip = [
    'growing region',
    'province',
    'department',
    'zone',
    'district',
    'county',
    'state',
    'region'
]

In [53]:
# Stripping function
def strip_suffixes(region):
    if pd.isna(region):
        return region
    for suffix in suffixes_to_strip:
        if region.endswith(suffix):
            return region.replace(suffix, '').strip()
    return region

In [54]:
# Apply function
df_coffee_wrangle['coffee_region'] = df_coffee_wrangle['coffee_region'].apply(strip_suffixes)

In [55]:
# Work check
df_coffee_wrangle['coffee_region'].value_counts(dropna=False).head(30)

coffee_region
unknown                  486
southern ethiopia        237
southcentral kenya       159
hawaii                   133
western panama           123
oromia                   120
southcentral ethiopia     72
cauca                     47
yirgacheffe               30
gedeo                     28
guji                      28
huila                     27
ethiopia                  27
central kenya             26
north sumatra             24
guatemala                 23
sumatra                   19
boquete                   18
colombia                  18
valle del cauca           17
sidamo                    14
jimma                     14
northern sumatra          14
sidama                    14
volcan                    14
antioquia                 13
kenya                     13
chiriqui                  13
southern colombia         12
central highlands         12
Name: count, dtype: int64

In [56]:
# spot audit 
df_coffee_wrangle[df_coffee_wrangle['coffee_origin'].str.contains('growing region|province|department|zone|district|county|state|region', case=False, na=False)][['coffee_origin', 'coffee_region']].head(30)

,coffee_origin,coffee_region
0,"quindio department, colombia",unknown
1,"quindio department, colombia",unknown
3,"gimbo, kaffa province, ethiopia",kaffa
4,"sidamo growing region, southern ethiopia",southern ethiopia
7,"huila department, colombia",unknown
10,"sidamo growing region, southern ethiopia",southern ethiopia
11,"caicendonia, valle del cauca, cauca department...",valle del cauca
12,"piendamo, cauca department, colombia",cauca
13,"mbozi district, songwe region, tanzania",songwe
14,"gedeb district, gedeo zone, southern ethiopia",gedeo


##### Strip Directional Prefixes

In [57]:
# Strip Directional Prefixes
def strip_directional_prefix(region):
    if pd.isna(region):
        return region
    directions = ['north', 'south', 'east', 'west', 'central']
    for d in directions:
        if region.startswith(d + ' '):
            return region.replace(d + ' ', '').strip()
    return region

In [58]:
# Apply the stripping
df_coffee_wrangle['coffee_region'] = df_coffee_wrangle['coffee_region'].apply(strip_directional_prefix)

In [59]:
# Work check
df_coffee_wrangle['coffee_region'].value_counts(dropna=False).head(30)

coffee_region
unknown                  486
southern ethiopia        237
southcentral kenya       159
hawaii                   133
western panama           123
oromia                   120
southcentral ethiopia     72
cauca                     47
sumatra                   43
kenya                     39
guatemala                 31
yirgacheffe               30
guji                      28
gedeo                     28
huila                     27
ethiopia                  27
colombia                  25
boquete                   18
valle del cauca           17
sidamo                    14
sidama                    14
jimma                     14
volcan                    14
northern sumatra          14
chiriqui                  13
antioquia                 13
southern colombia         12
highlands                 12
valley                    11
quindio                   11
Name: count, dtype: int64

In [60]:
# Spot audit
df_coffee_wrangle[
    df_coffee_wrangle['coffee_origin'].str.contains('south|north|east|west|central', case=False, na=False)
][['coffee_origin', 'coffee_region']].head(30)

,coffee_origin,coffee_region
4,"sidamo growing region, southern ethiopia",southern ethiopia
10,"sidamo growing region, southern ethiopia",southern ethiopia
14,"gedeb district, gedeo zone, southern ethiopia",gedeo
15,"guji zone, oromia region, southern ethiopia",oromia
17,"alajuela, central valley, costa rica",valley
18,"guji growing region, south-central ethiopia",southcentral ethiopia
20,"sidamo growing region, southern ethiopia",southern ethiopia
23,"yirgacheffe growing region, southern ethiopia",southern ethiopia
25,"kirinyaga district, south-central kenya",southcentral kenya
32,"sidamo growing region, southern ethiopia",southern ethiopia


In [62]:
# Final directional prefixes
direction_strip_map = {
    'southern ethiopia': 'ethiopia',
    'southcentral ethiopia': 'ethiopia',
    'southcentral kenya': 'kenya',
    'southern colombia': 'colombia',
    'northern sumatra': 'sumatra',
    'western panama': 'panama',
    'central highlands': 'highlands',
    'central valley': 'unknown',
    'valley': 'unknown'
}

In [63]:
# Strip the remaining directional prefixes
def strip_directional_variant(region):
    return direction_strip_map.get(region, region)

df_coffee_wrangle['coffee_region'] = df_coffee_wrangle['coffee_region'].apply(strip_directional_variant)

In [64]:
# Work check
df_coffee_wrangle['coffee_region'].value_counts(dropna=False).head(30)

coffee_region
unknown               497
ethiopia              336
kenya                 198
hawaii                133
panama                124
oromia                120
sumatra                57
cauca                  47
colombia               37
guatemala              31
yirgacheffe            30
guji                   28
gedeo                  28
huila                  27
boquete                18
valle del cauca        17
sidama                 14
sidamo                 14
jimma                  14
volcan                 14
antioquia              13
chiriqui               13
highlands              12
quindio                11
chimaltenango          10
far western panama      9
valle de cauca          9
costa rica              9
huehuetenango           8
loja                    8
Name: count, dtype: int64

In [65]:
# Spot Aduit
df_coffee_wrangle[
    df_coffee_wrangle['coffee_origin'].str.contains('south|north|east|west|central', case=False, na=False)
][['coffee_origin', 'coffee_region']].head(30)

,coffee_origin,coffee_region
4,"sidamo growing region, southern ethiopia",ethiopia
10,"sidamo growing region, southern ethiopia",ethiopia
14,"gedeb district, gedeo zone, southern ethiopia",gedeo
15,"guji zone, oromia region, southern ethiopia",oromia
17,"alajuela, central valley, costa rica",unknown
18,"guji growing region, south-central ethiopia",ethiopia
20,"sidamo growing region, southern ethiopia",ethiopia
23,"yirgacheffe growing region, southern ethiopia",ethiopia
25,"kirinyaga district, south-central kenya",kenya
32,"sidamo growing region, southern ethiopia",ethiopia


##### Country to Unknown

In [67]:
# Convert any country in region list to unknown.
known_countries = sorted([
    'australia', 'brazil', 'burundi', 'canada', 'china', 'colombia', 'costa rica',
    'el salvador', 'england', 'ethiopia', 'guatemala', 'guinea', 'honduras',
    'hong kong', 'india', 'indonesia', 'japan', 'kenya', 'macao', 'mexico',
    'nicaragua', 'panama', 'peru', 'puerto rico', 'rwanda', 'south korea',
    'taiwan', 'tanzania', 'timor', 'united kingdom', 'united states', 'yemen'
])

def fallback_country_to_unknown(region):
    if region in known_countries:
        return 'unknown'
    return region
# Apply
df_coffee_wrangle['coffee_region'] = df_coffee_wrangle['coffee_region'].apply(fallback_country_to_unknown)

In [68]:
# Work check
df_coffee_wrangle['coffee_region'].value_counts(dropna=False).head(30)

coffee_region
unknown               1260
hawaii                 133
oromia                 120
sumatra                 57
cauca                   47
yirgacheffe             30
guji                    28
gedeo                   28
huila                   27
boquete                 18
valle del cauca         17
jimma                   14
sidama                  14
sidamo                  14
volcan                  14
chiriqui                13
antioquia               13
highlands               12
quindio                 11
chimaltenango           10
valle de cauca           9
far western panama       9
huehuetenango            8
loja                     8
pichincha                7
minas gerais             7
caldas                   7
tarraz                   6
tolima                   6
sanaa governorate        6
Name: count, dtype: int64

In [69]:
# Spot aduit
df_coffee_wrangle[
    df_coffee_wrangle['coffee_origin'].str.contains(
        'kenya|ethiopia|colombia|guatemala|panama|peru|mexico|rwanda|ecuador|indonesia|honduras|el salvador|brazil|yemen|tanzania|burundi|congo|costa rica|united states|united kingdom|china|india|japan|hong kong|macao|puerto rico|nicaragua|south korea|taiwan|guinea|canada|england|australia|timor',
        case=False, na=False)
][['coffee_origin', 'coffee_region']].head(30)

,coffee_origin,coffee_region
0,"quindio department, colombia",unknown
1,"quindio department, colombia",unknown
2,colombia; ethiopia,unknown
3,"gimbo, kaffa province, ethiopia",kaffa
4,"sidamo growing region, southern ethiopia",unknown
5,panama; ethiopia,unknown
6,"yunlin, taiwan",unknown
7,"huila department, colombia",unknown
8,"cusco, peru",unknown
9,"yunlin, taiwan",unknown


## 7. Wrangle Roaster Region

##### Hawaii Normalization

In [72]:
# Strip unicode from roaster_region
df_coffee_wrangle['roaster_region'] = df_coffee_wrangle['roaster_region'].apply(strip_unicode)

In [73]:
# Preview work
df_coffee_wrangle['roaster_region'].value_counts(dropna=False).head(30)

roaster_region
unknown           672
california        305
wisconsin         241
virginia          115
minnesota         113
hawaii            100
colorado           72
massachusetts      66
kansas             52
illinois           48
new jersey         37
montana            37
washington         36
new york           27
north carolina     26
connecticut        24
indiana            24
oregon             19
new hampshire      17
nevada             16
florida            14
ontario            12
vermont            10
ohio                9
georgia             9
oklahoma            8
texas               7
maine               7
missouri            6
hawaii island       6
Name: count, dtype: int64

In [74]:
# Work check
df_coffee_wrangle[['roaster_location', 'roaster_region']].head(50)

,roaster_location,roaster_region
0,"cleveland, tennessee",tennessee
1,"cleveland, tennessee",tennessee
2,"jersey city, new jersey",new jersey
3,"chia-yi, taiwan",unknown
4,"chia-yi, taiwan",unknown
5,"taichung, taiwan",unknown
6,"yunlin, taiwan",unknown
7,"chia-yi, taiwan",unknown
8,"chia-yi, taiwan",unknown
9,"yunlin, taiwan",unknown


In [75]:
# Pull Unique roaster_region Values
unique_roaster_regions = sorted(df_coffee_wrangle['roaster_region'].dropna().unique().tolist())
for region in unique_roaster_regions:
    print(region)

alaska
alberta
arizona
arkansas
big island of hawaii
british columbia
calfornia
california
californiaa
chiayi
colorado
connecticut
dc
florida
georgia
hawaii
hawaii island
idaho
illinois
indiana
iowa
kansas
kentucky
korea
louisiana
maine
maryland
massachusetts
maui
michigan
minnesota
mississippi
missouri
montana
nantou
nevada
new hampshire
new jersey
new mexico
new york
north carolina
oaxaca
ohio
oklahoma
ontario
oregon
pennsylvania
saskatchewan
satipo province
south dakota
sydney
taichung
taiwan
tennessee
texas
unknown
vermont
virginia
virginia and floyd
washington
wisconsin
wyoming
yunnan province


In [77]:
# Filter hawaii-related regions
hawaii_candidates = [
    region for region in unique_roaster_regions
    if any(keyword in region for keyword in ['hawaii', 'kona', 'kau', 'puna', 'lahaina', 'maui', 'big island'])
]

In [78]:
# Build hawaii normalization map
hawaii_map = {variant: 'hawaii' for variant in hawaii_candidates}
list(hawaii_map.keys())

['big island of hawaii', 'hawaii', 'hawaii island', 'maui']

In [79]:
# Apply hawaii normalization
def normalize_hawaii(region):
    return hawaii_map.get(region, region)

df_coffee_wrangle['roaster_region'] = df_coffee_wrangle['roaster_region'].apply(normalize_hawaii)

In [80]:
# Work check
df_coffee_wrangle['roaster_region'].value_counts(dropna=False).head(30)

roaster_region
unknown           672
california        305
wisconsin         241
virginia          115
hawaii            113
minnesota         113
colorado           72
massachusetts      66
kansas             52
illinois           48
new jersey         37
montana            37
washington         36
new york           27
north carolina     26
indiana            24
connecticut        24
oregon             19
new hampshire      17
nevada             16
florida            14
ontario            12
vermont            10
georgia             9
ohio                9
oklahoma            8
maine               7
texas               7
missouri            6
mississippi         5
Name: count, dtype: int64

In [81]:
# Spot Audit
df_coffee_wrangle[
    df_coffee_wrangle['roaster_region'].str.contains('hawaii|kona|kau|puna|lahaina|maui|big island', case=False, na=False)
    & (df_coffee_wrangle['roaster_region'] != 'hawaii')
][['roaster_location', 'roaster_region']].head(30)

,roaster_location,roaster_region


##### Strip Suffixes

In [82]:
unique_roaster_regions = sorted(df_coffee_wrangle['roaster_region'].dropna().unique().tolist())
for region in unique_roaster_regions:
    print(region)

alaska
alberta
arizona
arkansas
british columbia
calfornia
california
californiaa
chiayi
colorado
connecticut
dc
florida
georgia
hawaii
idaho
illinois
indiana
iowa
kansas
kentucky
korea
louisiana
maine
maryland
massachusetts
michigan
minnesota
mississippi
missouri
montana
nantou
nevada
new hampshire
new jersey
new mexico
new york
north carolina
oaxaca
ohio
oklahoma
ontario
oregon
pennsylvania
saskatchewan
satipo province
south dakota
sydney
taichung
taiwan
tennessee
texas
unknown
vermont
virginia
virginia and floyd
washington
wisconsin
wyoming
yunnan province


In [83]:
roaster_region_map = {
    'virginia and floyd': 'virginia',
    'satipo province': 'satipo',
    'yunnan province': 'yunnan'
}

In [84]:
df_coffee_wrangle['roaster_region'] = df_coffee_wrangle['roaster_region'].replace(roaster_region_map)

In [85]:
df_coffee_wrangle['roaster_region'].value_counts(dropna=False).head(30)

roaster_region
unknown           672
california        305
wisconsin         241
virginia          116
hawaii            113
minnesota         113
colorado           72
massachusetts      66
kansas             52
illinois           48
new jersey         37
montana            37
washington         36
new york           27
north carolina     26
connecticut        24
indiana            24
oregon             19
new hampshire      17
nevada             16
florida            14
ontario            12
vermont            10
georgia             9
ohio                9
oklahoma            8
maine               7
texas               7
missouri            6
mississippi         5
Name: count, dtype: int64

##### Country to Unknown

In [86]:
# List of known countries
known_countries = sorted([
    'australia', 'brazil', 'burundi', 'canada', 'china', 'colombia', 'costa rica',
    'el salvador', 'england', 'ethiopia', 'guatemala', 'guinea', 'honduras',
    'hong kong', 'india', 'indonesia', 'japan', 'kenya', 'macao', 'mexico',
    'nicaragua', 'panama', 'peru', 'puerto rico', 'rwanda', 'south korea',
    'taiwan', 'tanzania', 'timor', 'united kingdom', 'united states', 'yemen',
    'korea'
])

In [87]:
# Convert to unknown
def fallback_country_to_unknown(region):
    if region in known_countries:
        return 'unknown'
    return region

df_coffee_wrangle['roaster_region'] = df_coffee_wrangle['roaster_region'].apply(fallback_country_to_unknown)

In [88]:
# Work check
df_coffee_wrangle['roaster_region'].value_counts(dropna=False).head(30)

roaster_region
unknown           675
california        305
wisconsin         241
virginia          116
hawaii            113
minnesota         113
colorado           72
massachusetts      66
kansas             52
illinois           48
new jersey         37
montana            37
washington         36
new york           27
north carolina     26
connecticut        24
indiana            24
oregon             19
new hampshire      17
nevada             16
florida            14
ontario            12
vermont            10
ohio                9
georgia             9
oklahoma            8
maine               7
texas               7
missouri            6
mississippi         5
Name: count, dtype: int64

##### Correct Spelling

In [89]:
# Look at the roaster region list
unique_roaster_regions = sorted(df_coffee_wrangle['roaster_region'].dropna().unique().tolist())
for region in unique_roaster_regions:
    print(region)

alaska
alberta
arizona
arkansas
british columbia
calfornia
california
californiaa
chiayi
colorado
connecticut
dc
florida
georgia
hawaii
idaho
illinois
indiana
iowa
kansas
kentucky
louisiana
maine
maryland
massachusetts
michigan
minnesota
mississippi
missouri
montana
nantou
nevada
new hampshire
new jersey
new mexico
new york
north carolina
oaxaca
ohio
oklahoma
ontario
oregon
pennsylvania
saskatchewan
satipo
south dakota
sydney
taichung
tennessee
texas
unknown
vermont
virginia
washington
wisconsin
wyoming
yunnan


In [91]:
# Correct spellings
spelling_map = {
    'calfornia': 'california',
    'californiaa': 'california',
    'dc': 'washington dc'
}
# Apply
df_coffee_wrangle['roaster_region'] = df_coffee_wrangle['roaster_region'].replace(spelling_map)

In [94]:
# Work check
unique_roaster_regions = sorted(df_coffee_wrangle['roaster_region'].dropna().unique().tolist())
for region in unique_roaster_regions:
    print(region)

alaska
alberta
arizona
arkansas
british columbia
california
chiayi
colorado
connecticut
florida
georgia
hawaii
idaho
illinois
indiana
iowa
kansas
kentucky
louisiana
maine
maryland
massachusetts
michigan
minnesota
mississippi
missouri
montana
nantou
nevada
new hampshire
new jersey
new mexico
new york
north carolina
oaxaca
ohio
oklahoma
ontario
oregon
pennsylvania
saskatchewan
satipo
south dakota
sydney
taichung
tennessee
texas
unknown
vermont
virginia
washington
washington dc
wisconsin
wyoming
yunnan


In [96]:
df_coffee_wrangle.head()

,coffee_name,total_score,roast_level,agtron_roast,price_usd,quantity_g,usd_per_gram,roaster_location,roaster_city,roaster_region,roaster_country,coffee_origin,coffee_city,coffee_region,coffee_country
0,colombia finca campo hermosa,94.0,light,82.0,29.99,226.8,$29.99 / 226.8g,"cleveland, tennessee",cleveland,tennessee,united states,"quindio department, colombia",quindio department,unknown,colombia
1,colombia finca la sirena mango co-ferment,94.0,light,87.0,22.99,226.8,$22.99 / 226.8g,"cleveland, tennessee",cleveland,tennessee,united states,"quindio department, colombia",quindio department,unknown,colombia
2,in bloom,94.0,light,88.0,25.0,250.0,$25.0 / 250.0g,"jersey city, new jersey",jersey city,new jersey,united states,colombia; ethiopia,colombia,unknown,colombia
3,ethiopia washed kaffa gimbo lot rich espresso,96.0,medium light,77.0,8.17,226.8,$8.17 / 226.8g,"chia-yi, taiwan",chia-yi,unknown,taiwan,"gimbo, kaffa province, ethiopia",gimbo,kaffa,ethiopia
4,ethiopia natural gute bona,95.0,medium light,78.0,13.07,226.8,$13.07 / 226.8g,"chia-yi, taiwan",chia-yi,unknown,taiwan,"sidamo growing region, southern ethiopia",sidamo growing region,unknown,unknown


## 8. Export

In [ ]:
# Export wrangled dataset
df_coffee_wrangle.to_csv(
    r'C:\Users\Chase\anaconda_projects\Exercise_6_Coffee\A6_Coffee\02_Data\Prepared_Data\Top Rated Coffee\top_rated_coffee_wrangled.csv',
    index=False,
    encoding='utf-8'
)